

**Assignment 4: Training a DCGAN on the Fashion-MNIST Dataset**

**The big idea:**
- The **Generator (G)** starts with random noise and tries to turn it into a fake clothing image.
- The **Discriminator (D)** looks at an image (real or fake) and guesses whether it's real or fake.
- They train against each other: G tries to fool D, D tries not to be fooled. Over time G gets better at making realistic images.

Run each cell top to bottom (Runtime > Run all works too). On Colab, go to **Runtime > Change runtime type > GPU** first, this will train much faster.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

os.makedirs("grids_simple", exist_ok=True)

## Step 1: Settings

Kept small on purpose for a first, understandable run. Increase `EPOCHS` to 25-30 for your final submission once you've confirmed everything works.

In [ ]:
NOISE_SIZE = 64      # length of the random noise vector fed into G
IMAGE_SIZE = 32       # we resize 28x28 images to 32x32 (easier for the math)
BATCH_SIZE = 128
EPOCHS = 30           # assignment asks for at least 25-30 epochs
SAVE_EVERY = 5

## Step 2: Load the data

Fashion-MNIST = 60,000 grayscale images of clothes (28x28 pixels). We normalize pixel values to [-1, 1] because the Generator will output values in that same range (`tanh` activation).

In [ ]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

data = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
loader = torch.utils.data.DataLoader(data, batch_size=BATCH_SIZE, shuffle=True)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.78MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.1MB/s]


## Step 3: The Generator

Think of it as: noise -> tiny image -> bigger image -> bigger image -> final 32x32 image. Each `ConvTranspose2d` layer doubles the image size.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # Input: noise vector shaped (NOISE_SIZE, 1, 1)
            # Output: 4x4 image with 128 channels
            nn.ConvTranspose2d(NOISE_SIZE, 128, kernel_size=4, stride=1, padding=0),
            nn.BatchNorm2d(128),   # keeps training stable
            nn.ReLU(),

            # 4x4 -> 8x8
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # 8x8 -> 16x16
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # 16x16 -> 32x32, output 1 channel (grayscale)
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()   # squashes output to [-1, 1], matching our normalized data
        )

    def forward(self, noise):
        return self.net(noise)

## Step 4: The Discriminator

Basically the reverse of the Generator: takes an image, shrinks it down step by step, and outputs a single number between 0 and 1 (1 = "I think this is real", 0 = "I think this is fake").

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 32x32 -> 16x16
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),   # like ReLU but allows small negative values

            # 16x16 -> 8x8
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),

            # 8x8 -> 4x4
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            # 4x4 -> 1x1 (a single decision number)
            nn.Conv2d(128, 1, kernel_size=4, stride=1, padding=0),
            nn.Sigmoid()   # squashes output to [0, 1] (a probability)
        )

    def forward(self, image):
        return self.net(image).view(-1)

In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)

## Step 5: Loss function and optimizers

BCELoss = Binary Cross Entropy — standard choice for "real vs fake" (yes/no) style problems.

In [ ]:
loss_function = nn.BCELoss()

optimizer_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Same noise every time we save a grid, so we can watch the SAME
# fake images improve over epochs instead of seeing random new ones.
fixed_noise = torch.randn(16, NOISE_SIZE, 1, 1, device=device)

## Step 6: Training loop

Each batch, we do two separate updates:
1. Update D: show it real images (label=1) and fake images (label=0)
2. Update G: try to make D output 1 (real) for its fake images

A 4x4 grid of generated images is saved every `SAVE_EVERY` epochs, using the same fixed noise each time so you can watch the same samples evolve.

In [ ]:
g_losses = []
d_losses = []

for epoch in range(1, EPOCHS + 1):
    total_g_loss = 0
    total_d_loss = 0

    for real_images, _ in loader:
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        real_labels = torch.ones(batch_size, device=device)
        fake_labels = torch.zeros(batch_size, device=device)

        # ---------- Train Discriminator ----------
        optimizer_D.zero_grad()

        # How well does D recognize real images as real?
        prediction_real = D(real_images)
        loss_real = loss_function(prediction_real, real_labels)

        # Make some fake images and see if D correctly calls them fake
        noise = torch.randn(batch_size, NOISE_SIZE, 1, 1, device=device)
        fake_images = G(noise)
        prediction_fake = D(fake_images.detach())  # detach: do not train G here
        loss_fake = loss_function(prediction_fake, fake_labels)

        d_loss = loss_real + loss_fake
        d_loss.backward()
        optimizer_D.step()

        # ---------- Train Generator ----------
        optimizer_G.zero_grad()

        # G wants D to say these fake images are "real" (label=1)
        prediction = D(fake_images)
        g_loss = loss_function(prediction, real_labels)
        g_loss.backward()
        optimizer_G.step()

        total_d_loss += d_loss.item()
        total_g_loss += g_loss.item()

    avg_d = total_d_loss / len(loader)
    avg_g = total_g_loss / len(loader)
    d_losses.append(avg_d)
    g_losses.append(avg_g)
    print(f"Epoch {epoch}/{EPOCHS} | D loss: {avg_d:.3f} | G loss: {avg_g:.3f}")

    # Save a 4x4 grid of sample images every few epochs
    if epoch % SAVE_EVERY == 0 or epoch == 1:
        with torch.no_grad():
            samples = G(fixed_noise).cpu()
        grid = vutils.make_grid(samples, nrow=4, normalize=True)
        vutils.save_image(grid, f"grids_simple/epoch_{epoch}.png")

        plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
        plt.title(f"Epoch {epoch}")
        plt.axis("off")
        plt.show()

## Step 7: Plot the losses

In [ ]:
plt.plot(g_losses, label="Generator loss")
plt.plot(d_losses, label="Discriminator loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training Losses")
plt.savefig("grids_simple/loss_plot.png")
plt.show()

## Step 8: Written observations

Fill this in after inspecting your saved grids in `grids_simple/` and the loss plot above.

- **Early epochs (1-5):** Generated images are mostly random noise/blobs, no clear structure yet. D loss is typically low (easily tells real from fake) while G loss is high.
- **Middle epochs (10-20):** Rough silhouettes of clothing items (shirts, trousers, shoes) start to emerge, along with blur/artifacts. Losses usually start to oscillate as G and D compete more evenly.
- **Later epochs (25-30):** Shapes become sharper and more consistent. Check whether improvement has plateaued (losses flattening out) or whether you see:
  - **Mode collapse**: the generator repeatedly produces very similar-looking images regardless of the input noise.
  - **Instability**: loss spikes or oscillations that don't settle.
- **Note:** In GANs, loss values don't necessarily decrease monotonically the way they do in supervised learning — a stable oscillation between G and D losses is often a sign of healthy adversarial training, not a bug. If D's loss crashes to near-zero and stays there, D is "winning" too easily and G will stop learning.